# Uncertainty Approximation Methods — Comparison

Comparison of uncertainty estimation methods across 3 model families on regression problems.

**Models:**
- Bayesian Linear Regression with radial basis features
- Gaussian Process
- Neural Network (MLP)

**Uncertainty methods:**
- Exact Bayesian inference (BLR and GP — natively)
- Monte Carlo Dropout (Neural Network)
- Deep Ensembles (Neural Network)

This notebook keeps almost no logic itself — it imports model definitions,
methods and plotting helpers from `src/`. The actual code lives in the
repository so it can be reused across multiple notebooks (one per dataset
later, plus a summary notebook).

**How to use**
- The first cell clones the repo into the Colab runtime.
- To switch datasets: comment/uncomment the appropriate `DATASET = ...` line.
- Models are trained from scratch when `MODE = "train"` and saved to
  `SAVE_ROOT/{DATASET}/...`. Re-run with `MODE = "load"` to skip training.

## 0. Repo setup

Clones the project repository into the Colab runtime and adds it to `sys.path`
so that `src/` is importable. **Edit `REPO_URL` below to point to your fork.**

In [ ]:
import os, sys

REPO_URL = "https://github.com/MichalKocwa/ml-uncertainty-msc.git"
REPO_DIR = "/content/uncertainty-thesis"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Auto-reload modules from src/ when files change (useful when iterating)
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data import load_dataset
from src.models import (
    MLP, make_mlp, train_mlp, mlp_point_predict,
    BayesianRBFRegression, make_gp,
)
from src.methods import mc_dropout_predict, train_ensemble, ensemble_predict
from src.persistence import (
    set_save_dir, save_sklearn, load_sklearn,
    save_nn, load_nn, save_nn_ensemble, load_nn_ensemble,
)
from src.plotting import plot_uncertainty

np.random.seed(42)
torch.manual_seed(42)

## 1. Persistence setup

`MODE`:
- `"train"` — train all models and save to `SAVE_ROOT/{DATASET}/...`.
- `"load"` — skip training and load previously saved artifacts.

Artifacts go to Google Drive by default. Falls back to local folder if not on Colab.

In [ ]:
MODE = "train"              # "train" | "load"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    SAVE_ROOT = "/content/drive/MyDrive/uncertainty_models"
except (ImportError, ModuleNotFoundError):
    SAVE_ROOT = "./uncertainty_models"

print(f"MODE      = {MODE}")
print(f"SAVE_ROOT = {SAVE_ROOT}")

## 2. Dataset

Pick a dataset by uncommenting one line below. Every dataset gets its own
subdirectory under `SAVE_ROOT`, so models for different datasets do not
overwrite each other.

In [ ]:
DATASET = "sin"
# DATASET = "concrete"
# DATASET = "power_plant"
# DATASET = "protein"

X_train, y_train, X_eval, y_eval, is_1d = load_dataset(DATASET)

set_save_dir(os.path.join(SAVE_ROOT, DATASET))

print(f"Dataset:    {DATASET}")
print(f"  Train:    {X_train.shape[0]} samples, dim={X_train.shape[1]}")
print(f"  Eval:     {X_eval.shape[0]} {'grid points' if is_1d else 'test samples'}")

if is_1d:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.scatter(X_train, y_train, s=20, label="train", color="C0")
    ax.plot(X_eval, y_eval, "k--", alpha=0.4, label="true function")
    ax.set_title(f"Dataset: {DATASET}")
    ax.legend()
    plt.show()

In [ ]:
def show(y_pred, y_std, title):
    """Tiny wrapper so model cells stay short."""
    plot_uncertainty(X_train, y_train, X_eval, y_eval, y_pred, y_std, is_1d, title)

## 3. Bayesian Linear Regression with radial basis features

A parametric Bayesian model. Inputs are mapped to radial basis features around
a random subset of training points, then Bayesian linear regression on those
features gives a closed-form Gaussian predictive distribution. This is the
parametric counterpart to the Gaussian Process — same flavor of uncertainty,
finite-dimensional weight space.

In [ ]:
if MODE == "train":
    blr = BayesianRBFRegression(n_centers=50).fit(X_train, y_train)
    save_sklearn(blr, "blr")
    print(f"Trained + saved BLR. gamma={blr.gamma_:.4g}, n_centers={len(blr.centers_)}")
else:
    blr = load_sklearn("blr")
    print(f"Loaded BLR. gamma={blr.gamma_:.4g}, n_centers={len(blr.centers_)}")

y_pred, y_std = blr.predict(X_eval, return_std=True)
show(y_pred, y_std, "Bayesian Linear Regression (RBF features) — exact posterior")

## 4. Gaussian Process

GP is a non-parametric Bayesian model with a closed-form Gaussian posterior
predictive. Reference method: uncertainty comes directly from exact Bayesian
inference, no approximation.

In [ ]:
if MODE == "train":
    gp = make_gp().fit(X_train, y_train)
    save_sklearn(gp, "gp")
    print(f"Trained + saved GP. Fitted kernel: {gp.kernel_}")
else:
    gp = load_sklearn("gp")
    print(f"Loaded GP. Fitted kernel: {gp.kernel_}")

y_pred, y_std = gp.predict(X_eval, return_std=True)
show(y_pred, y_std, "Gaussian Process — exact posterior")

## 5. Neural Network — Monte Carlo Dropout

Train a single MLP with dropout, then keep dropout active at inference time
and take many stochastic forward passes (Gal & Ghahramani 2016).

In [ ]:
IN_DIM = X_train.shape[1]

if MODE == "train":
    nn_dropout = train_mlp(X_train, y_train, dropout=0.2, epochs=1000, seed=42)
    save_nn(nn_dropout, "nn_dropout")
else:
    nn_dropout = load_nn(lambda: make_mlp(IN_DIM, dropout=0.2), "nn_dropout")

y_pred, y_std = mc_dropout_predict(nn_dropout, X_eval, n_samples=200)
show(y_pred, y_std, "Neural Network — Monte Carlo Dropout (200 samples)")

## 6. Neural Network — Deep Ensemble

Train several MLPs from different random initializations and aggregate their
predictions (Lakshminarayanan et al. 2017).

In [ ]:
N_NN = 5

if MODE == "train":
    nn_ensemble = train_ensemble(X_train, y_train, n_models=N_NN, epochs=1000)
    save_nn_ensemble(nn_ensemble, "nn_ensemble")
else:
    nn_ensemble = load_nn_ensemble(lambda: make_mlp(IN_DIM, dropout=0.0), "nn_ensemble")

y_pred, y_std = ensemble_predict(nn_ensemble, X_eval)
show(y_pred, y_std, f"Neural Network — Deep Ensemble (N={len(nn_ensemble)})")

## 7. What to look for

**On 1D synthetic data (`sin`):** training spans `x ∈ [0, 6]`, everything outside is OOD.
- *BLR (RBF features):* uncertainty grows smoothly outside `[0, 6]` because RBF features decay there → posterior over weights cannot constrain the prediction.
- *Gaussian Process:* same qualitative behavior, kernel "knows" test points are far from training data.
- *Deep Ensemble:* members disagree strongly in the OOD region → wide band there.
- *Monte Carlo Dropout:* OOD growth is typically smaller than ensembles — known limitation.

**On multivariate datasets:** OOD is no longer obvious from a 1D plot. Read the diagnostics as:
- *Predicted vs true:* points should cluster around the diagonal; error bars should cover the diagonal for ~95% of points if the method is well-calibrated at 2σ.
- *Sorted predictions:* the band should be tighter where predictions are confident and wider where they aren't. Constant-width bands indicate the method is not adapting to input difficulty.

## Next steps

1. Add quantitative metrics: NLL, CRPS, PICP@95%, MPIW, calibration error, reliability diagrams (see `uncertainty-toolbox` library).
2. Cross-dataset comparison table.
3. Add Laplace approximation as a third method for the neural network.